Forward Fill (ffill) with GroupBy
===
Difficulty: Medium

Problem Description:
===================
You're given a DataFrame with `client_id`, `revenue`, and `date`. Some `revenue` values are NaN
because no transaction occurred. Fill each NaN with the **previous non-null revenue for the same
client** (forward fill per group).

Sample Input:
```
| client_id | date       | revenue |
|-----------|------------|---------|
| A         | 2021-01-01 | 100     |
| A         | 2021-01-02 | NaN     |
| A         | 2021-01-03 | 200     |
| B         | 2021-01-01 | NaN     |
| B         | 2021-01-02 | 50      |
```

Sample Output:
```
| client_id | date       | revenue |
|-----------|------------|---------|
| A         | 2021-01-01 | 100     |
| A         | 2021-01-02 | 100     |  ← ffill from Jan 1
| A         | 2021-01-03 | 200     |
| B         | 2021-01-01 | NaN     |  ← no previous value for B
| B         | 2021-01-02 | 50      |
```

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'client_id': ['A','A','A','B','B','B','C','C'],
    'date':      pd.to_datetime(['2021-01-01','2021-01-02','2021-01-03',
                                 '2021-01-01','2021-01-02','2021-01-03',
                                 '2021-01-01','2021-01-02']),
    'revenue':   [100, np.nan, 200, np.nan, 50, np.nan, np.nan, 300]
})
print(df)

**Concepts to use:**
1. **`groupby().ffill()`** — forward fill within each group independently (critical: global `ffill()` would bleed Client A's values into Client B).
2. **Why not global `fillna(method='ffill')`?** — it ignores group boundaries; a leading NaN in group B would pick up the last value from group A.
3. **`sort_values` first** — always sort by `[client_id, date]` before filling to ensure chronological order.

In [ ]:
# Optimised Solution
def forward_fill_by_group(df):
    df = df.sort_values(['client_id', 'date']).copy()
    # ffill within each client group — does NOT bleed across groups
    df['revenue'] = df.groupby('client_id')['revenue'].ffill()
    return df

print(forward_fill_by_group(df))

# Common mistake to avoid:
# df['revenue'].fillna(method='ffill')  ← global ffill, bleeds across clients!